In [1]:
import pandas as pd

warning_levels = pd.read_excel("../data/raw/Kosi_Warning_Level_Government_Data.xlsx")

In [2]:
station_files = {
    "Balanga": "../data/raw/data_Balanga.xlsx",       # kept, but will end up unlabeled
    "Baltara": "../data/raw/data_Baltara.xlsx",
    "Basua": "../data/raw/data_Basua.xlsx",
    "Birpur": "../data/raw/data_Birpur.xlsx",
    "Jainagar": "../data/raw/data_Jainagar.xlsx",
    "Kursela": "../data/raw/data_Kursela.xlsx",
}

all_stations = []
for station_name, path in station_files.items():
    df = pd.read_excel(path)
    df["time"] = pd.to_datetime(df["time"])
    df["station_clean"] = station_name
    all_stations.append(df)

combined = pd.concat(all_stations, ignore_index=True)
print(combined.shape)
combined.head()

(170628, 6)


,Unnamed: 0,time,name,WSE,code,station_clean
0,0,2020-01-01 08:00:00,Balanga,3.32,NaN,Balanga
1,1,2020-01-01 13:00:00,Balanga,3.32,NaN,Balanga
2,2,2020-01-01 18:00:00,Balanga,3.32,NaN,Balanga
3,3,2020-01-02 08:00:00,Balanga,3.32,NaN,Balanga
4,4,2020-01-02 13:00:00,Balanga,3.32,NaN,Balanga


In [3]:
name_mapping = {
    "Birpur (0 km)": "Birpur",
    "Basua (64.25 km d/s)": "Basua",
    "Baltara": "Baltara",
    "Kursela": "Kursela",
    "Jainagar (Kamla)": "Jainagar",
}

warning_levels["station_clean"] = warning_levels["Station"].map(name_mapping)

# Sanity check — should show exactly 5 matched rows
warning_levels[warning_levels["station_clean"].notna()][
    ["Station", "station_clean", "Danger Level (m)", "Warning Level (m)", "HFL / Highest Flood Level (m)"]
]

,Station,station_clean,Danger Level (m),Warning Level (m),HFL / Highest Flood Level (m)
3,Birpur (0 km),Birpur,74.70,73.70,76.11
5,Basua (64.25 km d/s),Basua,47.75,46.75,49.24
8,Baltara,Baltara,33.85,32.85,36.40
10,Kursela,Kursela,30.00,29.00,32.10
14,Jainagar (Kamla),Jainagar,68.50,67.50,71.35


In [4]:
merged = combined.merge(
    warning_levels[["station_clean", "Danger Level (m)", "Warning Level (m)", "HFL / Highest Flood Level (m)"]],
    on="station_clean",
    how="left"
)

print("Rows before merge:", len(combined))
print("Rows after merge:", len(merged))
print("Rows with thresholds (labelable):", merged["Danger Level (m)"].notna().sum())
print("Rows without thresholds (Balanga, unlabeled):", merged["Danger Level (m)"].isna().sum())

Rows before merge: 170628
Rows after merge: 170628
Rows with thresholds (labelable): 145171
Rows without thresholds (Balanga, unlabeled): 25457


In [5]:
def label_risk(row):
    wse = row["WSE"]
    warning = row["Warning Level (m)"]
    danger = row["Danger Level (m)"]
    hfl = row["HFL / Highest Flood Level (m)"]

    if pd.isna(wse) or pd.isna(danger):
        return "unknown"
    elif wse >= hfl:
        return "extreme"
    elif wse >= danger:
        return "danger"
    elif wse >= warning:
        return "warning"
    else:
        return "normal"

merged["risk_label"] = merged.apply(label_risk, axis=1)
print(merged["risk_label"].value_counts())

risk_label
normal     66677
danger     40691
warning    37785
unknown    25457
extreme       18
Name: count, dtype: int64


In [6]:
labeled = merged[merged["risk_label"] != "unknown"].copy()
labeled["risk_label"] = labeled["risk_label"].replace("extreme", "danger")

print(labeled.shape)
print(labeled["risk_label"].value_counts())

(145171, 10)
risk_label
normal     66677
danger     40709
warning    37785
Name: count, dtype: int64


In [7]:
print(labeled["station_clean"].value_counts())
print()
print(labeled.isna().sum())

station_clean
Jainagar    29498
Baltara     29031
Basua       29028
Kursela     28948
Birpur      28666
Name: count, dtype: int64

Unnamed: 0                           0
time                                 0
name                                 0
WSE                                  0
code                             25468
station_clean                        0
Danger Level (m)                     0
Warning Level (m)                    0
HFL / Highest Flood Level (m)        0
risk_label                           0
dtype: int64


In [8]:
import os
os.makedirs("../data/processed", exist_ok=True)
labeled.to_parquet("../data/processed/labeled_stations.parquet", index=False)
print("Saved:", labeled.shape)

Saved: (145171, 10)
